计算指标

In [1]:
# ===================== TUSCAN 计算并保存 ARI 指标 =====================
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import os
import warnings
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore")

# ==========================================
# 1. 修正数据路径
# ==========================================
TUSCAN_CSV_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/TUSCAN_Results/Human_breast_cancer/TUSCAN_ClusterLabels.csv"
H5AD_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad" 

# ==========================================
# 2. 读取与清洗数据
# ==========================================
print("===== 加载与对齐数据 =====")
adata = ad.read_h5ad(H5AD_PATH)
tuscan_labels = pd.read_csv(TUSCAN_CSV_PATH, index_col="Spot_Barcode")

# 还原 R 语言修改的 Barcode 格式 (将 '.' 替换回 '-')
tuscan_labels.index = tuscan_labels.index.str.replace('.', '-', regex=False)

# 合并标签
adata.obs = adata.obs.join(tuscan_labels, how="left")

# 检查匹配成功的 Spot 数量
matched_spots = adata.obs["TUSCAN_Tumor_Normal"].notna().sum()
print(f"🔗 成功匹配到 TUSCAN 预测结果的 Spot 数量: {matched_spots} / {adata.n_obs}")

if matched_spots == 0:
    raise ValueError("❌ 匹配失败！参与计算的 Spot 数量为 0，请检查 CSV 中的 Barcode 格式。")

adata = adata[adata.obs["TUSCAN_Tumor_Normal"].notna()].copy()

# ==========================================
# 3. 标签映射 (二分类：Invasive+Tumor vs Healthy+Surrounding tumor)
# ==========================================
# 真实标签映射
gt_map_int = {
    'Tumor': 1,
    'Invasive': 1,
    'Surrounding tumor': 0,
    'Healthy': 0
}

pred_map = {'Tumor': 1, 'Normal': 0}

gt_labels_for_math = adata.obs['annot_type'].map(gt_map_int).values

# 映射预测标签
if adata.obs['TUSCAN_Tumor_Normal'].dtype == object or adata.obs['TUSCAN_Tumor_Normal'].dtype.name == 'category':
    pred_labels = adata.obs['TUSCAN_Tumor_Normal'].map(pred_map).values
else:
    pred_labels = adata.obs['TUSCAN_Tumor_Normal'].values

# ==========================================
# 4. 计算 ARI
# ==========================================
valid_mask = ~np.isnan(gt_labels_for_math) & ~np.isnan(pred_labels)
y_true = gt_labels_for_math[valid_mask]
y_pred = pred_labels[valid_mask]

print(f"📊 最终有效参与 ARI 计算的 Spot 数量: {len(y_true)}")

if len(y_true) > 0:
    ari_val = adjusted_rand_score(y_true, y_pred)
    print("\n" + "="*50)
    print(f"✅ TUSCAN 模型乳腺癌分割 ARI 指标：{ari_val:.4f}")
    print("="*50)
    
    # ==========================================
    # 5. 保存结果到指定目录
    # ==========================================
    SAVE_DIR = "/home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_breast_cancer_ViHBC/TUSCAN"
    os.makedirs(SAVE_DIR, exist_ok=True)
    csv_save_path = os.path.join(SAVE_DIR, "TUSCAN_ARI.csv")

    results_df = pd.DataFrame({
        "Metric Name": ["Adjusted Rand Index"],
        "Abbreviation": ["ARI"],
        "Value": [round(ari_val, 4)],
        "Ideal Trend": ["Closer to 1"],
        "Evaluation Dimension": ["Macro Boundary (Supervised)"]
    })

    results_df.to_csv(csv_save_path, index=False, encoding="utf-8-sig")

    print(f"\n🎉 TUSCAN 评估结果已成功保存至：\n   {csv_save_path}")
    display(results_df)

else:
    print("❌ 有效计算数据为空，无法得出真实 ARI。")

===== 加载与对齐数据 =====
🔗 成功匹配到 TUSCAN 预测结果的 Spot 数量: 3798 / 3798
📊 最终有效参与 ARI 计算的 Spot 数量: 3798

✅ TUSCAN 模型乳腺癌分割 ARI 指标：0.4339

🎉 TUSCAN 评估结果已成功保存至：
   /home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_breast_cancer_ViHBC/TUSCAN/TUSCAN_ARI.csv


,Metric Name,Abbreviation,Value,Ideal Trend,Evaluation Dimension
0,Adjusted Rand Index,ARI,0.4339,Closer to 1,Macro Boundary (Supervised)


In [ ]:
# ===================== TUSCAN 专属指标评估代码（无SpaCET，纯TUSCAN结果） =====================
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import os
import warnings
from sklearn.metrics import (
    adjusted_rand_score, 
    f1_score, 
    silhouette_score, 
    davies_bouldin_score
)

warnings.filterwarnings("ignore")

# ==========================================
# 【固定配置】你提供的文件路径（直接使用）
# ==========================================
TUSCAN_CSV_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/TUSCAN_Results/Human_breast_cancer/TUSCAN_ClusterLabels.csv"
H5AD_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_ViHBC_NicheCompassAndmy_Final.h5ad"
# 输出保存路径
SAVE_DIR = "/home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_Prostate_Cancer/TUSCAN"

# ==========================================
# 1. 读取数据：h5ad + TUSCAN分类结果
# ==========================================
print("===== 1. 加载数据 =====")
# 读取空间转录组h5ad
adata = ad.read_h5ad(H5AD_PATH)
# 读取TUSCAN肿瘤/正常分类结果
tuscan_labels = pd.read_csv(TUSCAN_CSV_PATH, index_col="Spot_Barcode")

# 将TUSCAN标签合并到adata（按Spot条码匹配）
adata.obs = adata.obs.join(tuscan_labels, how="left")
# 过滤掉标签缺失的Spot
adata = adata[adata.obs["TUSCAN_Tumor_Normal"].notna()].copy()

print(f"✅ 数据加载完成 | 有效Spot数量：{adata.n_obs}")
print(f"✅ TUSCAN分类统计：\n{adata.obs['TUSCAN_Tumor_Normal'].value_counts()}")

# ==========================================
# 2. 标签映射：Ground Truth（病理注释）+ TUSCAN预测标签
# 【乳腺癌病理标签映射】和原代码保持一致
# ==========================================
gt_map_int = {
    'Tumor': 1,
    'Invasive': 1,
    'Surrounding tumor': 0,
    'Healthy': 0
}

gt_map_str = {
    'Tumor': 'Tumor_Region',
    'Invasive': 'Tumor_Region',
    'Surrounding tumor': 'Non_Tumor_Region',
    'Healthy': 'Non_Tumor_Region'
}

adata.obs['Ground_Truth_Binary'] = adata.obs['annot_type'].map(gt_map_str).astype('category')
gt_labels_for_math = adata.obs['annot_type'].map(gt_map_int).values

# TUSCAN预测标签（核心！直接用TUSCAN结果：Tumor=1，Normal=0）
adata.obs['TUSCAN_Pred_Label'] = adata.obs['TUSCAN_Tumor_Normal'].map({'Tumor': 1, 'Normal': 0})
pred_labels = adata.obs['TUSCAN_Pred_Label'].values

# ==========================================
# 3. 空间可视化：真实标签 vs TUSCAN预测结果
# ==========================================
print("\n===== 2. 绘制空间分布对比图 =====")
try:
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    sc.pl.spatial(adata, color='Ground_Truth_Binary', title='Ground Truth (Pathologist)', spot_size=170, ax=axs[0], show=False)
    sc.pl.spatial(adata, color='TUSCAN_Tumor_Normal', title='TUSCAN Predicted Tumor/Normal', spot_size=170, ax=axs[1], show=False)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"可视化失败: {e}")

# ==========================================
# 4. 全维度指标计算（仅基于TUSCAN结果）
# ==========================================
print("\n" + "="*60)
print("🚀 开始执行 TUSCAN 全维度指标评估...")
print("="*60)

# --- 模块1：监督指标（真实标签 VS TUSCAN预测）：ARI、F1 ---
valid_mask = ~np.isnan(gt_labels)
y_true = gt_labels[valid_mask]
y_pred = pred_labels[valid_mask]

ari_val = adjusted_rand_score(y_true, y_pred)
f1_val = f1_score(y_true, y_pred, pos_label=1, average='binary')

print(f"🟢 [1/3] 宏观边界评估 (Supervised):")
print(f"  --> Adjusted Rand Index (ARI): {ari_val:.4f}")
print(f"  --> F1-Score (Tumor Region): {f1_val:.4f}")

# --- 模块2：无监督指标（聚类纯度）：ASW、DBI ---
print(f"\n🔵 [2/3] 微观特征纯度评估 (Unsupervised):")
# 计算PCA特征
if 'X_pca' not in adata.obsm:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.tl.pca(adata, svd_solver='arpack')

X_features = adata.obsm['X_pca']
asw_val = silhouette_score(X_features, pred_labels)
dbi_val = davies_bouldin_score(X_features, pred_labels)

print(f"  --> Average Silhouette Width (ASW): {asw_val:.4f}")
print(f"  --> Davies-Bouldin Index (DBI): {dbi_val:.4f}")

# --- 模块3：空间连贯性指标：Moran's I ---
print(f"\n🟣 [3/3] 物理空间连贯性指标 (Moran's I):")
if 'spatial_connectivities' not in adata.obsp:
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial", n_neighs=6)

# 计算TUSCAN预测结果的空间自相关性
tmp_adata = sc.AnnData(X=pred_labels.reshape(-1, 1).astype(float))
tmp_adata.obs_names = adata.obs_names
tmp_adata.var_names = ['TUSCAN_Tumor_Pred']
tmp_adata.obsp['spatial_connectivities'] = adata.obsp['spatial_connectivities']

sq.gr.spatial_autocorr(tmp_adata, mode="moran", genes=['TUSCAN_Tumor_Pred'], n_perms=100, n_jobs=-1)
moran_val = tmp_adata.uns["moranI"].loc['TUSCAN_Tumor_Pred', 'I']

print(f"  --> 总体平均 Moran's Index (Moran's I): {moran_val:.4f}")

# ==========================================
# 5. 保存指标结果
# ==========================================
os.makedirs(SAVE_DIR, exist_ok=True)
csv_save_path = os.path.join(SAVE_DIR, "评估结果_TUSCAN.csv")

results_df = pd.DataFrame({
    "Metric Name": [
        "Adjusted Rand Index", 
        "F1-Score", 
        "Average Silhouette Width", 
        "Davies-Bouldin Index", 
        "Moran's Index"
    ],
    "Abbreviation": ["ARI", "F1", "ASW", "DBI", "Moran's I"],
    "Value": [
        round(ari_val, 4), 
        round(f1_val, 4), 
        round(asw_val, 4), 
        round(dbi_val, 4), 
        round(moran_val, 4)
    ],
    "Ideal Trend": [
        "Closer to 1", "Closer to 1", "Closer to 1", 
        "Closer to 0", "Closer to 1 (>0)"
    ],
    "Evaluation Dimension": [
        "Macro Boundary (Supervised)", 
        "Macro Boundary (Supervised)", 
        "Micro Pureness (Unsupervised)", 
        "Micro Pureness (Unsupervised)", 
        "Spatial Topology (Physical)"
    ]
})

results_df.to_csv(csv_save_path, index=False, encoding="utf-8-sig")

# ==========================================
# 最终输出
# ==========================================
print("\n" + "="*60)
print(f"🎉 TUSCAN 指标计算完成！结果已保存至：\n   {csv_save_path}")
print("="*60)
print("\n指标汇总表：")
print(results_df)